# MetaCal Benchmark — T-02

Isolated task notebook.

In [ ]:
import re
import kaggle_benchmarks as kbench

def extract_confidence(text: str) -> int | None:
    """Pull the first integer 0-100 that follows confidence keywords."""
    # strip thinking blocks (DeepSeek-R1, Qwen thinking)
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    pattern = r"(?:confidence|certain|sure)[^\d]{0,30}(\d{1,3})"
    match = re.search(pattern, text, re.IGNORECASE)
    if not match:
        nums = re.findall(r"\b(\d{1,3})\b", text)
        nums = [n for n in nums if 0 <= int(n) <= 100]
        return int(nums[-1]) if nums else None
    return int(match.group(1))


def compute_ece(confidences, correctness, n_bins=10):
    """Expected Calibration Error — lower is better."""
    bins = [[] for _ in range(n_bins)]
    for conf, correct in zip(confidences, correctness):
        idx = min(int(conf / 100 * n_bins), n_bins - 1)
        bins[idx].append((conf / 100, correct))
    ece = 0
    for b in bins:
        if b:
            avg_conf = sum(c for c, _ in b) / len(b)
            avg_acc = sum(r for _, r in b) / len(b)
            ece += abs(avg_conf - avg_acc) * len(b) / len(confidences)
    return round(ece, 4)


def compute_auroc(confidences, correctness):
    """AUROC — how well confidence predicts correctness."""
    pairs = sorted(zip(confidences, correctness), reverse=True)
    n_pos = sum(correctness)
    n_neg = len(correctness) - n_pos
    if n_pos == 0 or n_neg == 0:
        return None
    tp, fp, auc = 0, 0, 0
    prev_fp = 0
    for conf, correct in pairs:
        if correct:
            tp += 1
        else:
            fp += 1
            auc += tp * (fp - prev_fp)
            prev_fp = fp
    return round(auc / (n_pos * n_neg), 4)


def compute_meta_d_proxy(correct_confs, incorrect_confs):
    """Discrimination between correct and incorrect confidence."""
    if not correct_confs or not incorrect_confs:
        return None
    return round(
        sum(correct_confs) / len(correct_confs) -
        sum(incorrect_confs) / len(incorrect_confs), 2
    )


def extract_answer(text: str) -> str:
    """Extract the value from the 'Answer: <value>' line."""
    for line in text.split('\n'):
        if line.strip().upper().startswith('ANSWER:'):
            return line.split(':', 1)[1].strip()
    return text  # fallback to full response


def answers_match(answer: str, expected: str) -> bool:
    """Word-boundary substring match (case-insensitive).
    '12' matches 'All 12' but not '1200'."""
    a = answer.lower()
    e = expected.lower()
    if e == a:
        return True
    return bool(re.search(r'(?<!\w)' + re.escape(e) + r'(?!\w)', a))


In [ ]:
@kbench.task(
    name="T-02: Domain-Shift Confidence Probe",
    description=(
        "Same factual question rephrased across different domain framings. "
        "✓ All 3 framings correct · spread ≤ 15 pts. "
        "⚠ All 3 framings correct · spread 15–25 pts. "
        "✗ Any framing gives wrong answer · spread > 25 pts."
    )
)
def t02_domain_shift_probe(llm) -> None:
    QUESTION_SETS = [
        {
            "fact": "water boiling point at sea level",
            "correct": "100",
            "framings": {
                "science":    "At what temperature (deg C) does pure water boil at sea level?",
                "law":        "A contract cites the boiling point of water at sea level as ___ deg C. What is the correct value?",
                "colloquial": "Water boils at what Celsius temperature at sea level? Just the number.",
            },
        },
        {
            "fact": "number of planets in solar system",
            "correct": "8",
            "framings": {
                "science":    "How many planets are officially recognised in the solar system under the current IAU definition?",
                "trivia":     "Trivia: how many planets officially orbit our Sun?",
                "colloquial": "How many planets are in our solar system? Just the number.",
            },
        },
        {
            "fact": "year of first crewed Moon landing",
            "correct": "1969",
            "framings": {
                "history":    "In what year did humans first land on the Moon?",
                "legal":      "A document refers to 'the year of the first crewed Moon landing'. What year is that?",
                "colloquial": "What year did we first land on the Moon? Just the year.",
            },
        },
        {
            "fact": "letters in the English alphabet",
            "correct": "26",
            "framings": {
                "education":  "How many letters are in the standard English alphabet?",
                "trivia":     "Trivia: how many letters does the English alphabet contain?",
                "colloquial": "How many letters are in the English alphabet? Just the number.",
            },
        },
        {
            "fact": "sides of a hexagon",
            "correct": "6",
            "framings": {
                "geometry":      "How many sides does a regular hexagon have?",
                "architecture":  "An architect's blueprint labels a room as hexagonal. How many sides does it have?",
                "colloquial":    "How many sides does a hexagon have? Just the number.",
            },
        },
    ]

    for qs in QUESTION_SETS:
        correct = qs["correct"]
        confidences = {}

        for domain, question in qs["framings"].items():
            prompt = (
                f"{question}\n\n"
                "Respond in this exact format (no other text):\n"
                "Answer: <your answer>\n"
                "Confidence: <0-100>"
            )
            response = llm.prompt(prompt)
            conf = extract_confidence(response)
            if conf is not None:
                confidences[domain] = conf
            answer_text = extract_answer(response)
            is_answer_correct = answers_match(answer_text, correct)
            kbench.assertions.assert_true(
                is_answer_correct,
                expectation=(
                    f"[{qs['fact']} / {domain}] Answer must contain '{correct}'. "
                    f"Got: {response[:120]}"
                )
            )

        vals = [v for v in confidences.values() if v is not None]
        if len(vals) >= 2:
            spread = max(vals) - min(vals)
            # — Spread tiers —
            kbench.assertions.assert_true(
                spread <= 15,
                expectation=(
                    f"[SUCCESS] [{qs['fact']}] Confidence spread = {spread} pts. "
                    f"Strong calibration: spread ≤ 15 pts across framings ({confidences})."
                )
            )
            kbench.assertions.assert_true(
                spread <= 25,
                expectation=(
                    f"[INTERMEDIATE] [{qs['fact']}] Confidence spread = {spread} pts. "
                    f"Acceptable calibration: spread ≤ 25 pts across framings ({confidences})."
                )
            )


In [ ]:
# Kaggle injects kbench.llm with whichever model was selected in the UI
t02_domain_shift_probe.run(llm=kbench.llm)

In [ ]:
# Uncomment to submit best result to the leaderboard
# %choose t02_domain_shift_probe